# 🚀 Data Lab Playground: the platform tour

**One shop, one pipeline.** Every notebook in this lab works on the same fictional company, **Northwind Gear**
(`lab_data.py`, derived from the dbt seeds). This notebook is the tour: it checks the services, runs the shop's numbers
through Spark, Iceberg, Hive and Trino, and shows how many rows each stage of the pipeline has produced so far.

## 🎯 **Platform components used here:**
- **🤖 AI & LLM**: any OpenAI-compatible LLM (or none: the AI sections then skip), LangChain, Phoenix observability
- **⚡ Data processing**: Apache Spark 4.1 (Spark Connect on Kubernetes)
- **💾 Storage**: SeaweedFS (S3), Iceberg tables (HMS Iceberg REST catalog) and plain Parquet (Hive catalog)
- **🔍 Query engine**: Trino, across the `iceberg` and `hive` catalogs

## 📝 **Sections:**
1–3. Environment, the LLM, Phoenix tracing
4. Northwind Gear: the pipeline map
5. Spark analysis with an AI summary
6. Bronze snapshot and a cross-catalog Trino join
7. Iceberg time travel and schema evolution
8. Rows per layer
9–11. More LangChain tracing, quick access guide, Spark on Kubernetes

## 1. Environment Setup and Service Health Check

In [ ]:
from lab_utils import PHOENIX_OTLP_ENDPOINT, llm_available, llm_provider
# Import required libraries
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import langchain

print(f"🤖 LLM provider: {llm_provider()}" + ("" if llm_available() else "  (AI sections are skipped; set the chart's llm values to run them)"))

# Phoenix setup with automatic LangChain instrumentation
phoenix_initialized = False
try:
    import phoenix as px
    from phoenix.otel import register

    # Configure Phoenix tracing endpoint
    phoenix_endpoint = PHOENIX_OTLP_ENDPOINT

    # Register Phoenix tracer for the default project
    tracer_provider = register(
        project_name="ai-platform-demo",
        endpoint=phoenix_endpoint
    )

    print("✅ Phoenix tracer registered successfully")
    print(f"📡 Traces will be sent to: {phoenix_endpoint}")
    print("🔍 Project: ai-platform-demo")

    # Auto-instrument LangChain if available
    try:
        from openinference.instrumentation.langchain import LangChainInstrumentor

        # Enable automatic tracing for all LangChain operations
        LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

        print("✅ LangChain auto-instrumentation enabled")
        phoenix_initialized = True

    except ImportError:
        print("⚠️ LangChain instrumentation not available - install openinference-instrumentation-langchain")

except ImportError:
    print("❌ Phoenix not installed")

print(f"\n📦 Available libraries:")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Langchain: {langchain.__version__}")
print(f"Python Environment: {sys.executable}")
print(f"Phoenix Tracing: {'✅ Enabled' if phoenix_initialized else '❌ Disabled'}")

## 2. The LLM (any OpenAI-compatible endpoint)

In [ ]:
import lab_utils as lu

llm_ready = lu.skip_unless_llm("list the models and chat with the LLM")
if llm_ready:
    try:
        print("🤖 Models at the endpoint:")
        for model in lu.openai_client().models.list().data[:20]:
            print(f"  • {model.id}")
    except Exception as e:   # not every OpenAI-compatible server lists its models
        print(f"(model list not available: {type(e).__name__})")

    def chat_with_llm(prompt, model=None):
        """One chat turn on the lab's LLM (the endpoint and key come from the chart)."""
        try:
            return lu._chat(model or lu.chat_model(), [{"role": "user", "content": prompt}])["message"]["content"]
        except RuntimeError as e:
            return f"LLM error: {e}"

    print("\n💬 Testing the LLM...")
    print(f"🤖 {lu.chat_model()} response:\n{chat_with_llm('Explain what a data lake is in 50 words or less.')}")

## 3. Phoenix AI Observability with LangChain Integration

In [ ]:
from lab_utils import PHOENIX_OTLP_ENDPOINT, PHOENIX_URL
# Phoenix observability status and LangChain integration test
try:
    phoenix_health = requests.get(f'{PHOENIX_URL}/health', timeout=10)
    if phoenix_health.status_code == 200:
        print("✅ Phoenix AI Observability is running")
        print(f"📡 OTLP Endpoint: {PHOENIX_OTLP_ENDPOINT}")
        print("🎯 LangChain tracing is ENABLED" if phoenix_initialized else "⚠️ LangChain tracing is not enabled - run the setup cell first")
        print("📊 Open the Phoenix UI (the `phoenix` route of the lab's Gateway), project: ai-platform-demo")
    else:
        print(f"⚠️ Phoenix health check returned: {phoenix_health.status_code}")
except Exception as e:
    print(f"❌ Phoenix connection error: {e}")

llm = None
if lu.skip_unless_llm("LangChain test chain (traced in Phoenix)"):
    try:
        from langchain_core.prompts import PromptTemplate

        llm = lu.langchain_chat(temperature=0.7)       # traced by the instrumentor above
        test_prompt = PromptTemplate(input_variables=["topic"], template="Explain {topic} in exactly 2 sentences.")
        test_chain = test_prompt | llm                  # LCEL chain
        print("\n🔄 Running traced LangChain operation...")
        result = test_chain.invoke({"topic": "machine learning"})
        print(f"✅ LangChain Response: {result.content}")
        if phoenix_initialized:
            print("🎉 The operation was traced: look for project ai-platform-demo in Phoenix")
    except Exception as e:
        print(f"⚠️ LangChain test error: {e}")

## 4. Northwind Gear: one shop, one pipeline
Every notebook in this lab works on the same fictional company, **Northwind Gear**, a European electronics shop.
Its system of record is the dbt seeds (`dbt/lakehouse_demo/seeds`: 20 customers, 10 products, 60 orders, payments,
refunds, product costs, sales regions). `lab_data.py` derives everything else from them, deterministically, so every
derived row points at a real customer, product or order.

| Stage | Notebook | What happens to the shop data | Capability |
|---|---|---|---|
| 0 Tour | this notebook | bronze snapshot, cross-catalog SQL, rows per layer | platform map |
| 1 Ingest | `iceberg/04` | order events streamed into `shop_bronze.order_events` | streaming merge-append, Parquet tuning |
| 2 Clickstream | `iceberg/02` | web sessions as `VARIANT` in `shop_v3.web_events` | format v3, funnel per product |
| 3 Privacy | `iceberg/03` | a customer is forgotten in `shop_v3.orders` | deletion vectors, row lineage, expiry |
| 4 Layout | `iceberg/01` | delivery points around hub cities | Hilbert vs Z-order clustering |
| 5 Model | `dbt/01–03` | seeds → staging → marts in `dbt_demo` | dbt on Trino + Iceberg |
| 6 Govern / ask | `semantic/01` | certified metrics over the marts | semantic layer over MCP (chart) |
| 7 Explain | `rag_demo` | refund rate per product + support tickets | RAG + metrics |
| 8 Operate | Airflow DAG `northwind_daily` | bronze upsert, quality checks, product KPIs | orchestration (chart) |
| 9 Observe | `ops/01_pipeline_agent` | the DAG runs and the Spark apps | Airflow + Spark History MCP (chart) |

Layers: raw = `dbt_demo_raw` (seeds), bronze = `shop_bronze` (format v2, Spark writes, Trino reads), `shop_v3` =
format-v3 feature tables, models = `dbt_demo`, gold = `shop_gold` (the Airflow DAG).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import pandas as pd
from pyspark.sql import functions as F
import lab_data
from lab_utils import get_spark, trino, CATALOG

spark = get_spark("northwind-tour")
print("Spark", spark.version)
customers = spark.createDataFrame(pd.DataFrame(lab_data.customers()))
products = spark.createDataFrame(pd.DataFrame(lab_data.products()))
orders = (spark.createDataFrame(pd.DataFrame(lab_data.orders()))
          .withColumn("order_ts", F.to_timestamp("order_ts"))
          .join(products.select("product_id", "unit_price"), "product_id")
          .withColumn("amount", F.col("quantity") * F.col("unit_price")))
for name, df in (("customers", customers), ("products", products), ("orders", orders)):
    df.createOrReplaceTempView(name)
    print(f"{name:10s} {df.count():>3} rows")

## 5. Spark: revenue by category and country, and an AI summary
Raw status values here are `placed` / `shipped` / `returned`; the dbt staging model maps them to
`open` / `fulfilled` / `returned` (`dbt/01`).

In [ ]:
by_category = spark.sql("""
    SELECT p.category, count(*) AS orders, sum(o.amount) AS gross_sales,
           round(avg(CASE WHEN o.status = 'returned' THEN 1.0 ELSE 0.0 END), 2) AS return_share
    FROM orders o JOIN products p USING (product_id)
    GROUP BY p.category ORDER BY gross_sales DESC""").toPandas()
by_country = spark.sql("""
    SELECT c.country, count(*) AS orders, sum(o.amount) AS gross_sales
    FROM orders o JOIN customers c USING (customer_id)
    GROUP BY c.country ORDER BY gross_sales DESC""").toPandas()
print(by_category.to_string(index=False), "\n")
print(by_country.to_string(index=False))

In [ ]:
summary_prompt = ("You are a retail analyst at Northwind Gear. In three short bullet points, say what stands out "
                  "in these numbers (gross sales and the share of orders returned):\n\n"
                  f"By category:\n{by_category.to_string(index=False)}\n\nBy country:\n{by_country.to_string(index=False)}")
if lu.skip_unless_llm("AI summary of the numbers above"):
    try:
        print(llm.invoke(summary_prompt).content)      # traced in Phoenix when tracing is on
    except Exception as e:  # the numbers above do not depend on the LLM
        print(f"AI summary skipped, the LLM is not reachable: {type(e).__name__}: {e}")

## 6. Bronze snapshot: Iceberg via Spark, plain Parquet via Hive, one Trino query over both
* `shop_bronze.orders`: the enriched orders as a format-v2 Iceberg table (HMS Iceberg REST catalog).
* `hive.default.shop_customers`: the customers as plain Parquet files in SeaweedFS, registered in the Hive catalog
  (HMS Thrift) as an external table.

Trino joins the two catalogs in one statement.

In [ ]:
NS = f"{CATALOG}.shop_bronze"
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NS}")
(orders.select("order_id", "customer_id", "product_id", "quantity", "unit_price", "amount", "order_ts", "status")
       .writeTo(f"{NS}.orders").using("iceberg").tableProperty("format-version", "2").createOrReplace())
first_snapshot = spark.sql(f"SELECT snapshot_id FROM {NS}.orders.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0][0]

CUSTOMERS_PATH = "s3a://warehouse/shop_raw/customers/"
customers.coalesce(1).write.mode("overwrite").parquet(CUSTOMERS_PATH)
trino("DROP TABLE IF EXISTS hive.default.shop_customers")      # external table: drops the metadata, keeps the files
trino(f"""CREATE TABLE hive.default.shop_customers (customer_id bigint, name varchar, email varchar, country varchar,
          plan varchar, signup_date varchar) WITH (external_location = '{CUSTOMERS_PATH}', format = 'PARQUET')""")

trino("""SELECT c.country, c.plan, count(*) AS orders, sum(o.amount) AS gross_sales
         FROM iceberg.shop_bronze.orders o
         JOIN hive.default.shop_customers c ON c.customer_id = o.customer_id
         GROUP BY 1, 2 ORDER BY gross_sales DESC LIMIT 8""")

## 7. Iceberg time travel and schema evolution
The marketing team wants to know which channel each order came from. Adding a column is a metadata-only change; the
snapshot before it still answers queries exactly as it did.

In [ ]:
spark.sql(f"ALTER TABLE {NS}.orders ADD COLUMN channel STRING")
spark.sql(f"UPDATE {NS}.orders SET channel = CASE WHEN order_id % 3 = 0 THEN 'mobile' ELSE 'web' END")
spark.sql(f"SELECT channel, count(*) AS orders FROM {NS}.orders GROUP BY channel").show()
print("columns at the first snapshot:", spark.sql(f"SELECT * FROM {NS}.orders VERSION AS OF {first_snapshot}").columns)
spark.sql(f"SELECT snapshot_id, operation, committed_at FROM {NS}.orders.snapshots ORDER BY committed_at").show(truncate=False)

## 8. Rows per layer
Each stage of the pipeline writes its own tables. Run the notebooks in any order; this cell shows what exists so
far (Trino reads every table here, the format-v3 ones included).

In [ ]:
LAYERS = [
    ("raw", "iceberg.dbt_demo_raw.raw_orders", "dbt/01"),
    ("bronze", "iceberg.shop_bronze.orders", "this notebook"),
    ("bronze", "hive.default.shop_customers", "this notebook"),
    ("bronze", "iceberg.shop_bronze.order_events", "iceberg/04 (stream) + northwind_daily (daily replay)"),
    ("bronze", "iceberg.shop_bronze.deliveries_hilbert", "iceberg/01"),
    ("v3", "iceberg.shop_v3.web_events", "iceberg/02"),
    ("v3", "iceberg.shop_v3.orders", "iceberg/03"),
    ("models", "iceberg.dbt_demo.fct_orders", "dbt/01"),
    ("models", "iceberg.dbt_demo.fct_refunds", "dbt/01"),
    ("gold", "iceberg.shop_gold.product_kpis", "Airflow DAG northwind_daily"),
]
rows = []
for layer, table, source in LAYERS:
    try:
        n = int(trino(f"SELECT count(*) AS n FROM {table}").n[0])
    except Exception:
        n = None
    rows.append({"layer": layer, "table": table, "rows": n if n is not None else "not built yet", "written by": source})
pd.DataFrame(rows)

## 9. More traced LangChain operations

In [ ]:
if lu.skip_unless_llm("more traced LangChain operations") and llm is not None and phoenix_initialized:
    from langchain_core.prompts import PromptTemplate

    print("🔄 Running multiple traced operations (project: ai-platform-demo)")
    try:
        analysis_chain = PromptTemplate(
            input_variables=["data_type", "metric"],
            template="As a data analyst, analyze {data_type} data focusing on {metric}. Provide 3 key insights in bullet points.") | llm
        print("\n📊 Trace 1: data analysis chain")
        print(str(analysis_chain.invoke({"data_type": "customer", "metric": "satisfaction scores"}).content)[:150], "...")

        sql_chain = PromptTemplate(
            input_variables=["table", "requirement"],
            template="Generate a SQL query for table '{table}' to {requirement}. Return only the SQL query.") | llm
        print("\n🗄️ Trace 2: SQL generation chain")
        print(sql_chain.invoke({"table": "customers", "requirement": "find average age by region"}).content)
        print("\n🔍 Open Phoenix, project ai-platform-demo, and click a trace for the full prompt and response.")
    except Exception as e:
        print(f"⚠️ Tracing demo error: {e}")
elif lu.llm_available():
    print("ℹ️ Phoenix tracing is not ready: run the setup cell and the Phoenix test cell first.")

## 10. Quick Access Guide & Phoenix Setup

In [ ]:
from lab_utils import PHOENIX_OTLP_ENDPOINT
print("🎉 AI-enhanced data platform tour")
print("\n🔧 Platform capabilities:")
print(f"  {'✅' if lu.llm_available() else '⏭️ (off)'} LLM inference (provider: {lu.llm_provider()})")
print(f"  {'✅' if phoenix_initialized else '❌'} AI observability with Phoenix (LangChain auto-traced)")
print("  ✅ Spark, Iceberg, Hive and Trino over the Northwind shop")

print("\n🌐 Access points: the lab's Gateway routes (`*.datalab.localhost:8080` on the workstation lab):")
print("  jupyter, trino, airflow, spark-history, phoenix, qdrant (and the local LLM when deployed)")
print(f"\n📡 Phoenix trace endpoint: {PHOENIX_OTLP_ENDPOINT}  (project: ai-platform-demo)")

## 11. Spark on Kubernetes (Spark Connect)

In [ ]:
# Spark on Kubernetes (Spark Connect).
# charts/data-lab runs one Spark Connect server (image: data-lab-playground/spark, Spark 4.1.3) as a Deployment;
# this notebook reaches it through SPARK_REMOTE (with the bearer token). The Kubeflow SDK is used in
# connect(base_url=...) mode; listing SparkConnect resources only works where the Spark Operator runs them.
import os

if not os.environ.get("SPARK_REMOTE"):
    print("SPARK_REMOTE is not set -> run this notebook in the lab's Jupyter; nothing to show.")
else:
    from kubeflow.spark import SparkClient, KubernetesBackendConfig
    from pyspark.sql import SparkSession

    try:
        # needs Kubernetes API access; charts/data-lab mounts no ServiceAccount token in Jupyter
        client = SparkClient(KubernetesBackendConfig(namespace=os.environ.get("POD_NAMESPACE")))
        sessions = list(client.list_sessions())
    except Exception as e:  # no kube config, no SparkConnect CRD, or this pod may not list it (charts/data-lab)
        client, sessions = None, []
        print(f"No SparkConnect resources to list ({type(e).__name__}); the server is a plain Deployment here.")
    for info in sessions:
        print(f"{info.name}: state={info.state} service={info.service_name} driver_pod={info.driver_pod_name}")

    if client is not None:
        spark_k8s = client.connect(base_url=os.environ["SPARK_REMOTE"])
    else:
        spark_k8s = SparkSession.builder.remote(os.environ["SPARK_REMOTE"]).getOrCreate()
    print("Connected, Spark version on the server:", spark_k8s.version)
    spark_k8s.range(5).show()
